In [48]:
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, StratifiedKFold

from opfython.models import SupervisedOPF
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, balanced_accuracy_score, matthews_corrcoef, average_precision_score

from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC as SupportVectorMachineClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.inspection import permutation_importance

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid

# Resampling removido: não faz parte do protocolo.

from scipy.stats import wilcoxon
import scipy.stats as st

import logging
opfython_logger = logging.getLogger("opfython")
opfython_logger.setLevel(logging.CRITICAL + 1)
opfython_logger.propagate = False
import warnings

In [49]:
df = pd.read_csv('NPHA-doctor-visits.csv')

In [50]:
df

,Number of Doctors Visited,Age,Phyiscal Health,Mental Health,Dental Health,Employment,Stress Keeps Patient from Sleeping,Medication Keeps Patient from Sleeping,Pain Keeps Patient from Sleeping,Bathroom Needs Keeps Patient from Sleeping,Uknown Keeps Patient from Sleeping,Trouble Sleeping,Prescription Sleep Medication,Race,Gender
0,3,2,4,3,3,3,0,0,0,0,1,2,3,1,2
1,2,2,4,2,3,3,1,0,0,1,0,3,3,1,1
2,3,2,3,2,3,3,0,0,0,0,1,3,3,4,1
3,1,2,3,2,3,3,0,0,0,1,0,3,3,4,2
4,3,2,3,3,3,3,1,0,0,0,0,2,3,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,2,2,2,2,2,3,0,0,0,1,0,3,3,1,1
710,3,2,2,2,2,2,1,0,0,0,1,2,3,1,2
711,3,2,4,2,3,3,0,0,0,0,0,3,3,1,1
712,3,2,3,1,3,3,1,0,1,1,1,3,3,1,2


In [51]:
df.shape

(714, 15)

In [52]:
df.isna().sum()

Number of Doctors Visited                     0
Age                                           0
Phyiscal Health                               0
Mental Health                                 0
Dental Health                                 0
Employment                                    0
Stress Keeps Patient from Sleeping            0
Medication Keeps Patient from Sleeping        0
Pain Keeps Patient from Sleeping              0
Bathroom Needs Keeps Patient from Sleeping    0
Uknown Keeps Patient from Sleeping            0
Trouble Sleeping                              0
Prescription Sleep Medication                 0
Race                                          0
Gender                                        0
dtype: int64

In [53]:
df

,Number of Doctors Visited,Age,Phyiscal Health,Mental Health,Dental Health,Employment,Stress Keeps Patient from Sleeping,Medication Keeps Patient from Sleeping,Pain Keeps Patient from Sleeping,Bathroom Needs Keeps Patient from Sleeping,Uknown Keeps Patient from Sleeping,Trouble Sleeping,Prescription Sleep Medication,Race,Gender
0,3,2,4,3,3,3,0,0,0,0,1,2,3,1,2
1,2,2,4,2,3,3,1,0,0,1,0,3,3,1,1
2,3,2,3,2,3,3,0,0,0,0,1,3,3,4,1
3,1,2,3,2,3,3,0,0,0,1,0,3,3,4,2
4,3,2,3,3,3,3,1,0,0,0,0,2,3,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,2,2,2,2,2,3,0,0,0,1,0,3,3,1,1
710,3,2,2,2,2,2,1,0,0,0,1,2,3,1,2
711,3,2,4,2,3,3,0,0,0,0,0,3,3,1,1
712,3,2,3,1,3,3,1,0,1,1,1,3,3,1,2


In [54]:
#histograma das features

# features = df.select_dtypes(include=['number']).columns

# for feature in features:
#     plt.figure(figsize=(8, 6))
#     sns.histplot(data=df, x=feature, hue='Number of Doctors Visited', kde=False, bins=50, palette="pastel")

#     plt.xlabel(feature)
#     plt.ylabel("Contagem")
#     plt.title(f"Distribuição da feature: {feature}")
#     plt.legend(title="Class")
#     plt.show()

In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 714 entries, 0 to 713
Data columns (total 15 columns):
 #   Column                                      Non-Null Count  Dtype
---  ------                                      --------------  -----
 0   Number of Doctors Visited                   714 non-null    int64
 1   Age                                         714 non-null    int64
 2   Phyiscal Health                             714 non-null    int64
 3   Mental Health                               714 non-null    int64
 4   Dental Health                               714 non-null    int64
 5   Employment                                  714 non-null    int64
 6   Stress Keeps Patient from Sleeping          714 non-null    int64
 7   Medication Keeps Patient from Sleeping      714 non-null    int64
 8   Pain Keeps Patient from Sleeping            714 non-null    int64
 9   Bathroom Needs Keeps Patient from Sleeping  714 non-null    int64
 10  Uknown Keeps Patient from Sleeping          714 n

In [56]:
df.describe()

,Number of Doctors Visited,Age,Phyiscal Health,Mental Health,Dental Health,Employment,Stress Keeps Patient from Sleeping,Medication Keeps Patient from Sleeping,Pain Keeps Patient from Sleeping,Bathroom Needs Keeps Patient from Sleeping,Uknown Keeps Patient from Sleeping,Trouble Sleeping,Prescription Sleep Medication,Race,Gender
count,714.000000,714.0,714.000000,714.000000,714.000000,714.000000,714.000000,714.000000,714.000000,714.000000,714.000000,714.000000,714.000000,714.000000,714.00000
mean,2.112045,2.0,2.794118,1.988796,3.009804,2.806723,0.247899,0.056022,0.218487,0.504202,0.417367,2.407563,2.829132,1.425770,1.55042
std,0.683441,0.0,0.900939,0.939928,1.361117,0.586582,0.432096,0.230126,0.413510,0.500333,0.493470,0.670349,0.546767,1.003896,0.49780
min,1.000000,2.0,-1.000000,-1.000000,-1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,1.000000,1.00000
25%,2.000000,2.0,2.000000,1.000000,2.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,3.000000,1.000000,1.00000
50%,2.000000,2.0,3.000000,2.000000,3.000000,3.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000,3.000000,1.000000,2.00000
75%,3.000000,2.0,3.000000,3.000000,4.000000,3.000000,0.000000,0.000000,0.000000,1.000000,1.000000,3.000000,3.000000,1.000000,2.00000
max,3.000000,2.0,5.000000,5.000000,6.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000,3.000000,5.000000,2.00000


In [57]:
# visualizing the distribution of the features
fig, axes = plt.subplots(3, 5, figsize=(20, 15))
axes = axes.ravel()

for i, column in enumerate(df.columns):
    df[column].value_counts().plot(kind='bar', ax=axes[i])
    axes[i].set_title(column)

plt.tight_layout()
plt.show()

/tmp/ipykernel_666002/806374595.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [58]:
df['Number of Doctors Visited'].describe()

count    714.000000
mean       2.112045
std        0.683441
min        1.000000
25%        2.000000
50%        2.000000
75%        3.000000
max        3.000000
Name: Number of Doctors Visited, dtype: float64

In [59]:
df['Age'].describe()

count    714.0
mean       2.0
std        0.0
min        2.0
25%        2.0
50%        2.0
75%        2.0
max        2.0
Name: Age, dtype: float64

In [60]:
plt.figure(figsize=(8, 6))
sns.boxplot(y='Number of Doctors Visited', data=df)
plt.title("Boxplot of the Number of Doctors Visited column")
plt.show()

/tmp/ipykernel_666002/716654087.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [61]:
Q1_UP = df['Number of Doctors Visited'].quantile(0.25)
Q2_UP = df['Number of Doctors Visited'].quantile(0.5)
Q3_UP = df['Number of Doctors Visited'].quantile(0.75)

lower_limit_UP = Q1_UP - 1.5 * (Q3_UP - Q1_UP)
upper_limit_UP = Q3_UP + 1.5 * (Q3_UP - Q1_UP)

print(f"Feature: Number of Doctors Visited")
print(f"- Q1: {Q1_UP:.2f}")
print(f"- Q2 (Mediana): {Q2_UP:.2f}")
print(f"- Q3: {Q3_UP:.2f}")
print(f"- Limite Inferior: {lower_limit_UP:.2f}")
print(f"- Limite Superior: {upper_limit_UP:.2f}")

Feature: Number of Doctors Visited
- Q1: 2.00
- Q2 (Mediana): 2.00
- Q3: 3.00
- Limite Inferior: 0.50
- Limite Superior: 4.50


Existe um viés na Classe 2. Portanto seria bom utilizar tecnica de  subamostragem para a classe super-representada.

In [62]:
# Calculate the class distribution
class_distribution = df['Number of Doctors Visited'].value_counts()
print(class_distribution)

Number of Doctors Visited
2    372
3    211
1    131
Name: count, dtype: int64


In [63]:
df_ = df.copy()

age_dict = { 1: "50-64", 2: "65-80" }
df_['Age'] = df_['Age'].map(age_dict)

race_dict = { 1: "White, Non-Hispanic", 2: "Black, Non-Hispanic", 3: "Other, Non-Hispanic", 4: "Hispanic", 5: "2+ Races, Non-Hispanic" }
df_['Race'] = df_['Race'].map(race_dict)

gender_dict = { 1: "Male", 2: "Female" }
df_['Gender'] = df_['Gender'].map(gender_dict)

plt.subplots(figsize=(20, 10))
for i, col in enumerate(['Age', 'Race' , 'Gender']):
    plt.subplot(1, 3, i + 1)

    x = df_[col].value_counts()
    plt.bar(x.index, x.values)
    plt.title('Distribution of Patient ' + col)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Count')

plt.show()

/tmp/ipykernel_666002/533527952.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [64]:
fig, axes = plt.subplots(1, 3, figsize=(20, 10))
for i, col in enumerate(['Age', 'Race', 'Gender']):
    x = df_[col].value_counts()
    axes[i].bar(x.index.astype(str), x.values, color=sns.color_palette("viridis", len(x)))
    axes[i].set_title('Distribution of Patient ' + col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    # Adicionar valores em cima das barras
    for j, v in enumerate(x.values):
        axes[i].text(j, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


/tmp/ipykernel_666002/3179644547.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [65]:
# Calculate the correlation matrix
corr_matrix = df.corr()

# Create a heatmap
plt.figure(figsize=(18, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True, fmt=".2f", linewidths=0.5)

# Add annotations to the heatmap
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        text = f"{corr_matrix.iloc[i, j]:.2f}"
        plt.text(j + 0.5, i + 0.5, text, ha='center', va='center', color='black')

plt.title("Correlation Matrix")
plt.show()

/tmp/ipykernel_666002/1832467298.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [66]:
# Target documentado: 1 consulta -> 0; 2/3 consultas -> 1.
df['Number of Doctors Visited'] = df['Number of Doctors Visited'].map({1: 0, 2: 1, 3: 1}).astype('int8')
class_counts = df['Number of Doctors Visited'].value_counts().sort_index()
class_percentages = df['Number of Doctors Visited'].value_counts(normalize=True).sort_index().mul(100)
print('Target: 0=0-1 consulta; 1=2+ consultas')
print(pd.DataFrame({'count': class_counts, 'percent': class_percentages}))

Target: 0=0-1 consulta; 1=2+ consultas
                           count    percent
Number of Doctors Visited                  
0                            131  18.347339
1                            583  81.652661


LEGADO: codificação do alvo desativada; pipeline final usa target binário explícito.

In [67]:
categorical_features = [c for c in df.columns if c != 'Number of Doctors Visited']
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=False)
df_encoded

,Number of Doctors Visited,Age_2,Phyiscal Health_-1,Phyiscal Health_1,Phyiscal Health_2,Phyiscal Health_3,Phyiscal Health_4,Phyiscal Health_5,Mental Health_-1,Mental Health_1,...,Prescription Sleep Medication_1,Prescription Sleep Medication_2,Prescription Sleep Medication_3,Race_1,Race_2,Race_3,Race_4,Race_5,Gender_1,Gender_2
0,1,True,False,False,False,False,True,False,False,False,...,False,False,True,True,False,False,False,False,False,True
1,1,True,False,False,False,False,True,False,False,False,...,False,False,True,True,False,False,False,False,True,False
2,1,True,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,True,False,True,False
3,0,True,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,True,False,False,True
4,1,True,False,False,False,True,False,False,False,False,...,False,False,True,True,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,1,True,False,False,True,False,False,False,False,False,...,False,False,True,True,False,False,False,False,True,False
710,1,True,False,False,True,False,False,False,False,False,...,False,False,True,True,False,False,False,False,False,True
711,1,True,False,False,False,False,True,False,False,False,...,False,False,True,True,False,False,False,False,True,False
712,1,True,False,False,False,True,False,False,False,True,...,False,False,True,True,False,False,False,False,False,True


In [68]:
df_encoded = df_encoded.reset_index(drop=True)

In [69]:
X = df_encoded.drop(columns=['Number of Doctors Visited'])
y = df_encoded['Number of Doctors Visited'].astype('int8')

In [70]:
X

,Age_2,Phyiscal Health_-1,Phyiscal Health_1,Phyiscal Health_2,Phyiscal Health_3,Phyiscal Health_4,Phyiscal Health_5,Mental Health_-1,Mental Health_1,Mental Health_2,...,Prescription Sleep Medication_1,Prescription Sleep Medication_2,Prescription Sleep Medication_3,Race_1,Race_2,Race_3,Race_4,Race_5,Gender_1,Gender_2
0,True,False,False,False,False,True,False,False,False,False,...,False,False,True,True,False,False,False,False,False,True
1,True,False,False,False,False,True,False,False,False,True,...,False,False,True,True,False,False,False,False,True,False
2,True,False,False,False,True,False,False,False,False,True,...,False,False,True,False,False,False,True,False,True,False
3,True,False,False,False,True,False,False,False,False,True,...,False,False,True,False,False,False,True,False,False,True
4,True,False,False,False,True,False,False,False,False,False,...,False,False,True,True,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,True,False,False,True,False,False,False,False,False,True,...,False,False,True,True,False,False,False,False,True,False
710,True,False,False,True,False,False,False,False,False,True,...,False,False,True,True,False,False,False,False,False,True
711,True,False,False,False,False,True,False,False,False,True,...,False,False,True,True,False,False,False,False,True,False
712,True,False,False,False,True,False,False,False,True,False,...,False,False,True,True,False,False,False,False,False,True


In [71]:
y

0      1
1      1
2      1
3      0
4      1
      ..
709    1
710    1
711    1
712    1
713    1
Name: Number of Doctors Visited, Length: 714, dtype: int8

In [72]:
# feature_importances = pd.DataFrame(model.feature_importances_,
#                                    index = X.columns,
#                                    columns=['importance']).sort_values('importance', ascending=False)

# # Plotting
# plt.figure(figsize=(10, 6))
# feature_importances.plot(kind='barh')

In [73]:
y

0      1
1      1
2      1
3      0
4      1
      ..
709    1
710    1
711    1
712    1
713    1
Name: Number of Doctors Visited, Length: 714, dtype: int8

In [74]:
colunas_lixo = [
    'Race_5', 'Uknown Keeps Patient from Sleeping_0', 'Prescription Sleep Medication_2',
    'Phyiscal Health_1', 'Mental Health_-1', 'Race_4', 'Dental Health_1',
    'Prescription Sleep Medication_-1', 'Dental Health_-1', 'Mental Health_5',
    'Phyiscal Health_-1', 'Age_2', 'Gender_1', 'Phyiscal Health_3', 'Phyiscal Health_5', 'Dental Health_3',
    'Race_2', 'Trouble Sleeping_3', 'Mental Health_3', 'Gender_2',
    'Employment_1', 'Trouble Sleeping_2', 'Dental Health_6', 'Trouble Sleeping_-1'
]

In [75]:
X = X.drop(columns=colunas_lixo)

In [76]:
print(f"Dimensões atuais: {X.shape}")

Dimensões atuais: (714, 25)


In [77]:
X_resampled = X.values
y_resampled = y.to_numpy()

In [78]:
y = pd.Series(y_resampled, name='Number of Doctors Visited').astype('int8')
class_distribution = y.value_counts().sort_index()
print("Class distribution (sem resampling):")
print(class_distribution)

Class distribution (sem resampling):
Number of Doctors Visited
0    131
1    583
Name: count, dtype: int64


In [79]:
ax = y.value_counts().plot(kind='bar', title='Number of Doctors Visited', color=sns.color_palette("viridis", len(y.value_counts())))
ax.set_xlabel('Classes')
ax.set_ylabel('Count')
for p in ax.patches:
    ax.text(p.get_x() + p.get_width()/2, p.get_height() + 0.5, str(int(p.get_height())), ha='center', fontweight='bold')
plt.show()


/tmp/ipykernel_666002/3165614967.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [80]:
scaler = StandardScaler()
X_escalonado = scaler.fit_transform(X)

In [81]:
pca = PCA(n_components=2)
componentes_principais = pca.fit_transform(X_escalonado)
df_pca = pd.DataFrame(data=componentes_principais, columns=['PC1', 'PC2'])
df_pca['Classe_Alvo'] = y

In [82]:
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='PC1', 
    y='PC2', 
    hue='Classe_Alvo', 
    palette='viridis',
    data=df_pca, 
    alpha=0.8,
    s=70
)

<Axes: xlabel='PC1', ylabel='PC2'>

In [83]:
sns.scatterplot(x='PC1', y='PC2', hue='Classe_Alvo', palette='viridis', data=df_pca, alpha=0.8, s=70)
expl_1 = pca.explained_variance_ratio_[0]*100
expl_2 = pca.explained_variance_ratio_[1]*100
plt.title('Gráfico de Componentes Principais em 2D (PCA Plot)', fontsize=15)
plt.xlabel(f"Componente Principal 1 ({expl_1:.2f}% de variância agrupada)")
plt.ylabel(f"Componente Principal 2 ({expl_2:.2f}% de variância agrupada)")
plt.legend(title='Classe', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

/tmp/ipykernel_666002/3556441085.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [84]:
pca_3d = PCA(n_components=3)
comps_3d = pca_3d.fit_transform(X_escalonado)
    
df_pca3 = pd.DataFrame(data=comps_3d, columns=['PC1', 'PC2', 'PC3'])
df_pca3['Classe_Alvo'] = y
    
# Matplotlib puro pro 3D 
fig3d = plt.figure(figsize=(12, 10))
ax = fig3d.add_subplot(111, projection='3d')

cores = ['purple', 'teal', 'gold'] 
classes_unicas = sorted(list(df_pca3['Classe_Alvo'].unique()))
    
# Plota classe por classe
for cls, cor in zip(classes_unicas, cores):
    subset = df_pca3[df_pca3['Classe_Alvo'] == cls]
    ax.scatter(subset['PC1'], subset['PC2'], subset['PC3'], 
                color=cor, label=f'Classe {cls}', s=60, alpha=0.9, edgecolor='black', linewidth=0.5)

e1_3d = pca_3d.explained_variance_ratio_[0]*100
e2_3d = pca_3d.explained_variance_ratio_[1]*100
e3_3d = pca_3d.explained_variance_ratio_[2]*100

ax.set_xlabel(f"PC1 ({e1_3d:.2f}%)", fontweight='bold', labelpad=10)
ax.set_ylabel(f"PC2 ({e2_3d:.2f}%)", fontweight='bold', labelpad=10)
ax.set_zlabel(f"PC3 ({e3_3d:.2f}%)", fontweight='bold', labelpad=10)
plt.title('Profundidade Dimensional Alvo (PCA 3D)', fontsize=16)
plt.legend(title='Target/Classe', loc='best')

# Opcional (rotacionar ângulo)
ax.view_init(elev=20, azim=135)
plt.tight_layout()
plt.show()


/tmp/ipykernel_666002/298286174.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [85]:
df = pd.DataFrame(X, columns=X.columns)

In [86]:
df_encoded.shape

(714, 50)

In [87]:
df

,Phyiscal Health_2,Phyiscal Health_4,Mental Health_1,Mental Health_2,Mental Health_4,Dental Health_2,Dental Health_4,Dental Health_5,Employment_2,Employment_3,...,Pain Keeps Patient from Sleeping_0,Pain Keeps Patient from Sleeping_1,Bathroom Needs Keeps Patient from Sleeping_0,Bathroom Needs Keeps Patient from Sleeping_1,Uknown Keeps Patient from Sleeping_1,Trouble Sleeping_1,Prescription Sleep Medication_1,Prescription Sleep Medication_3,Race_1,Race_3
0,False,True,False,False,False,False,False,False,False,True,...,True,False,True,False,True,False,False,True,True,False
1,False,True,False,True,False,False,False,False,False,True,...,True,False,False,True,False,False,False,True,True,False
2,False,False,False,True,False,False,False,False,False,True,...,True,False,True,False,True,False,False,True,False,False
3,False,False,False,True,False,False,False,False,False,True,...,True,False,False,True,False,False,False,True,False,False
4,False,False,False,False,False,False,False,False,False,True,...,True,False,True,False,False,False,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,True,False,False,True,False,True,False,False,False,True,...,True,False,False,True,False,False,False,True,True,False
710,True,False,False,True,False,True,False,False,True,False,...,True,False,True,False,True,False,False,True,True,False
711,False,True,False,True,False,False,False,False,False,True,...,True,False,True,False,False,False,False,True,True,False
712,False,False,True,False,False,False,False,False,False,True,...,False,True,False,True,True,False,False,True,True,False


In [88]:
x = df

In [89]:
x

,Phyiscal Health_2,Phyiscal Health_4,Mental Health_1,Mental Health_2,Mental Health_4,Dental Health_2,Dental Health_4,Dental Health_5,Employment_2,Employment_3,...,Pain Keeps Patient from Sleeping_0,Pain Keeps Patient from Sleeping_1,Bathroom Needs Keeps Patient from Sleeping_0,Bathroom Needs Keeps Patient from Sleeping_1,Uknown Keeps Patient from Sleeping_1,Trouble Sleeping_1,Prescription Sleep Medication_1,Prescription Sleep Medication_3,Race_1,Race_3
0,False,True,False,False,False,False,False,False,False,True,...,True,False,True,False,True,False,False,True,True,False
1,False,True,False,True,False,False,False,False,False,True,...,True,False,False,True,False,False,False,True,True,False
2,False,False,False,True,False,False,False,False,False,True,...,True,False,True,False,True,False,False,True,False,False
3,False,False,False,True,False,False,False,False,False,True,...,True,False,False,True,False,False,False,True,False,False
4,False,False,False,False,False,False,False,False,False,True,...,True,False,True,False,False,False,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
709,True,False,False,True,False,True,False,False,False,True,...,True,False,False,True,False,False,False,True,True,False
710,True,False,False,True,False,True,False,False,True,False,...,True,False,True,False,True,False,False,True,True,False
711,False,True,False,True,False,False,False,False,False,True,...,True,False,True,False,False,False,False,True,True,False
712,False,False,True,False,False,False,False,False,False,True,...,False,True,False,True,True,False,False,True,True,False


In [90]:
x = x.values

In [91]:
x

array([[False,  True, False, ...,  True,  True, False],
       [False,  True, False, ...,  True,  True, False],
       [False, False, False, ...,  True, False, False],
       ...,
       [False,  True, False, ...,  True,  True, False],
       [False, False,  True, ...,  True,  True, False],
       [False, False, False, ...,  True,  True, False]], shape=(714, 25))

# Protocolo experimental canônico
As células 0–45 são EDA histórica e não fornecem objetos ao treino. Este bloco recarrega os dados, fixa o protocolo e separa resultados corrigidos de resultados legados.


In [92]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    precision_recall_curve,
    roc_curve,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from opfython.models import SupervisedOPF
from xgboost import XGBClassifier

SEED = 42
TARGET_COLUMN = "Number of Doctors Visited"
TARGET_MAP = {1: 0, 2: 1, 3: 1}
TARGET_LABELS = {0: "0–1 consulta", 1: "2+ consultas"}
DATA_PATH = Path("src/NPHA-doctor-visits.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("NPHA-doctor-visits.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/src/NPHA-doctor-visits.csv")
assert DATA_PATH.exists(), "CSV não encontrado"
raw_data = pd.read_csv(DATA_PATH)
assert len(raw_data) == 714, f"n inesperado: {len(raw_data)}"
assert TARGET_COLUMN in raw_data.columns
assert set(raw_data[TARGET_COLUMN].dropna().unique()).issubset(TARGET_MAP)
y = raw_data[TARGET_COLUMN].map(TARGET_MAP).astype("int8")
assert set(y.unique()) == {0, 1}
X = raw_data.drop(columns=[TARGET_COLUMN]).copy()
class_counts = y.value_counts().sort_index()
class_percentages = y.value_counts(normalize=True).sort_index().mul(100)
print("Target:", TARGET_LABELS)
print(class_counts)
print(class_percentages)

outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
y_array = y.to_numpy()
fold_indices = []
for outer_fold, (train_index, test_index) in enumerate(outer_cv.split(X, y_array), 1):
    fold_indices.append({
        "outer_fold": outer_fold,
        "train_index": train_index.tolist(),
        "test_index": test_index.tolist(),
    })

param_grid_KNN = {
    "n_neighbors": [5, 10, 20],
    "weights": ["uniform", "distance"],
}
param_grid_DT = {
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [1, 5],
}
param_grid_RF = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, None],
    "min_samples_leaf": [1, 5],
}
param_grid_SVM = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
}
param_grid_MLP = {
    "hidden_layer_sizes": [(32,), (32, 16)],
    "alpha": [0.0001, 0.01],
    "max_iter": [500],
}
param_grid_LR = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs"],
    "max_iter": [1000],
}
param_grid_XGB = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
}
param_grid_NB = {"var_smoothing": [1e-11, 1e-9, 1e-7]}
param_grid_NC = {
    "metric": ["euclidean", "manhattan"],
    "shrink_threshold": [None, 0.1],
}
param_grid_OPF = {"distance": ["euclidean", "manhattan"]}
param_grid_AdaBoost = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.05, 0.1, 1.0],
}
param_grid_Dummy = {"strategy": ["most_frequent"]}


Target: {0: '0–1 consulta', 1: '2+ consultas'}
Number of Doctors Visited
0    131
1    583
Name: count, dtype: int64
Number of Doctors Visited
0    18.347339
1    81.652661
Name: proportion, dtype: float64


In [93]:
def build_preprocessor(frame):
    numeric = frame.select_dtypes(include=[np.number]).columns.tolist()
    categorical = [column for column in frame.columns if column not in numeric]
    transformers = []
    if numeric:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric,
            )
        )
    if categorical:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            )
        )
    return ColumnTransformer(transformers=transformers, remainder="drop")


def prevalence_weights(y_fit):
    values, counts = np.unique(np.asarray(y_fit), return_counts=True)
    frequencies = dict(zip(values.tolist(), counts.tolist()))
    assert set(frequencies) == {0, 1}
    return {
        0: len(y_fit) / (2 * frequencies[0]),
        1: len(y_fit) / (2 * frequencies[1]),
    }


def fit_mdi_selector(encoded_fit, feature_names, feature_origins, y_fit):
    auxiliary = RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight=None, n_jobs=1)
    auxiliary.fit(encoded_fit, y_fit)
    importances = np.asarray(auxiliary.feature_importances_, dtype=float)
    order = np.argsort(importances)[::-1]
    ordered_importance = importances[order]
    cumulative = np.cumsum(ordered_importance) / (ordered_importance.sum() or 1.0)
    prefix_size = np.searchsorted(cumulative, 0.80) + 1
    selected_mask = np.zeros(len(feature_names), dtype=bool)
    selected_mask[order[:prefix_size]] = True
    metadata = [
        {
            "feature_transformed": feature_names[index],
            "feature_origin": feature_origins[index],
            "importance": float(ordered_importance[position]),
            "cumulative": float(cumulative[position]),
            "selected": bool(position < prefix_size),
        }
        for position, index in enumerate(order)
    ]
    return selected_mask, metadata


def prepare_split(X_fit, y_fit, X_apply, use_mdi=False):
    preprocessor = build_preprocessor(X_fit)
    encoded_fit = preprocessor.fit_transform(X_fit)
    encoded_apply = preprocessor.transform(X_apply)
    feature_names = preprocessor.get_feature_names_out().tolist()
    record = {
        "selected_mask": None,
        "feature_names": feature_names,
        "importances": None,
        "cumulative": None,
        "feature_origin": feature_names,
        "mdi_records": [],
    }
    if not use_mdi:
        return preprocessor, encoded_fit, encoded_apply, record
    feature_origins = []
    for transformer_name, _, columns in preprocessor.transformers_:
        if transformer_name == "categorical":
            encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
            feature_origins.extend(
                origin for origin, categories in zip(columns, encoder.categories_)
                for _ in categories
            )
        else:
            feature_origins.extend(columns)
    selected_mask, mdi_records = fit_mdi_selector(
        encoded_fit, feature_names, feature_origins, y_fit
    )
    importances = np.asarray([item["importance"] for item in mdi_records], dtype=float)
    cumulative = np.cumsum(importances) / (importances.sum() or 1.0)
    prefix_size = np.searchsorted(cumulative, 0.80) + 1
    record.update(
        {
            "selected_mask": selected_mask.tolist(),
            "importances": [item["importance"] for item in mdi_records],
            "cumulative": [item["cumulative"] for item in mdi_records],
            "feature_origin": [item["feature_origin"] for item in mdi_records],
            "mdi_records": mdi_records,
        }
    )
    assert cumulative[prefix_size - 1] >= 0.80
    return preprocessor, encoded_fit[:, selected_mask], encoded_apply[:, selected_mask], record


def make_estimator(name, params, scenario="baseline", y_fit=None):
    applied = dict(params)
    if name == "KNN":
        return KNeighborsClassifier(**applied)
    if name == "DT":
        if scenario == "cost_sensitive":
            applied["class_weight"] = prevalence_weights(y_fit)
        applied["random_state"] = SEED
        return DecisionTreeClassifier(**applied)
    if name == "RF":
        if scenario == "cost_sensitive":
            applied["class_weight"] = prevalence_weights(y_fit)
        applied["random_state"] = SEED
        applied["n_jobs"] = 1
        return RandomForestClassifier(**applied)
    if name == "SVM":
        if scenario == "cost_sensitive":
            applied["class_weight"] = prevalence_weights(y_fit)
        base = SVC(**applied)
        return CalibratedClassifierCV(
            estimator=base,
            cv=5,
            method="sigmoid",
        )
    if name == "MLP":
        applied["random_state"] = SEED
        return MLPClassifier(**applied)
    if name == "LR":
        if scenario == "cost_sensitive":
            applied["class_weight"] = prevalence_weights(y_fit)
        return LogisticRegression(**applied)
    if name == "XGB":
        if scenario == "cost_sensitive":
            weights = prevalence_weights(y_fit)
            applied["scale_pos_weight"] = weights[1] / weights[0]
        applied["random_state"] = SEED
        applied["n_jobs"] = 1
        return XGBClassifier(**applied)
    if name == "NB":
        return GaussianNB(**applied)
    if name == "NC":
        return NearestCentroid(**applied)
    if name == "AdaBoost":
        applied["random_state"] = SEED
        return AdaBoostClassifier(**applied)
    if name == "Dummy":
        return DummyClassifier(**applied, random_state=SEED)
    if name == "OPF":
        return SupervisedOPF(**applied)
    raise KeyError(name)


def predict_with_score(model, X_apply):
    prediction = np.asarray(model.predict(X_apply))
    if hasattr(model, "predict_proba"):
        probabilities = np.asarray(model.predict_proba(X_apply))
        classes = np.asarray(model.classes_)
        score0 = probabilities[:, np.flatnonzero(classes == 0)[0]]
        score1 = probabilities[:, np.flatnonzero(classes == 1)[0]]
        return prediction, score0, score1, "predict_proba"
    if hasattr(model, "decision_function"):
        decision = np.asarray(model.decision_function(X_apply))
        if decision.ndim == 1:
            return prediction, -decision, decision, "decision_function"
        classes = np.asarray(model.classes_)
        return (
            prediction,
            decision[:, np.flatnonzero(classes == 0)[0]],
            decision[:, np.flatnonzero(classes == 1)[0]],
            "decision_function",
        )
    return prediction, None, None, "unavailable"


def binary_metrics(y_true, prediction, score0=None, score1=None):
    matrix = confusion_matrix(y_true, prediction, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    result = {
        "accuracy": accuracy_score(y_true, prediction),
        "precision_0": precision_score(y_true, prediction, labels=[0], average=None, zero_division=0)[0],
        "precision_1": precision_score(y_true, prediction, labels=[1], average=None, zero_division=0)[0],
        "recall_0": recall_score(y_true, prediction, labels=[0], average=None, zero_division=0)[0],
        "recall_1": recall_score(y_true, prediction, labels=[1], average=None, zero_division=0)[0],
        "f1_0": f1_score(y_true, prediction, labels=[0], average=None, zero_division=0)[0],
        "f1_1": f1_score(y_true, prediction, labels=[1], average=None, zero_division=0)[0],
        "f1_macro": f1_score(y_true, prediction, labels=[0, 1], average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, prediction, labels=[0, 1], average="weighted", zero_division=0),
        "precision_macro": precision_score(y_true, prediction, labels=[0, 1], average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, prediction, labels=[0, 1], average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "mcc": matthews_corrcoef(y_true, prediction),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "score_source": "unavailable" if score1 is None else "continuous",
        "auroc": roc_auc_score(y_true, score1) if score1 is not None else np.nan,
        "ap_0": average_precision_score(1 - np.asarray(y_true), score0) if score0 is not None else np.nan,
        "ap_1": average_precision_score(y_true, score1) if score1 is not None else np.nan,
    }
    return result


# Fase 1 — Baseline
Todas as features, Dummy e roster histórico.


In [94]:
roster = ["KNN", "DT", "RF", "SVM", "MLP", "LR", "XGB", "NB", "NC", "OPF", "AdaBoost"]
grids = {"KNN": param_grid_KNN, "DT": param_grid_DT, "RF": param_grid_RF, "SVM": param_grid_SVM, "MLP": param_grid_MLP, "LR": param_grid_LR, "XGB": param_grid_XGB, "NB": param_grid_NB, "NC": param_grid_NC, "OPF": param_grid_OPF, "AdaBoost": param_grid_AdaBoost}
phase1_oof = {name: np.full(len(y_array), -1, dtype=int) for name in roster + ["Dummy"]}
phase1_scores = {name: np.full(len(y_array), np.nan) for name in roster + ["Dummy"]}
phase1_rows = []
phase1_search = []
for outer_fold, (train_index, test_index) in enumerate(outer_cv.split(X, y_array), 1):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y_array[train_index]
    y_test = y_array[test_index]
    for name in roster + ["Dummy"]:
        grid = param_grid_Dummy if name == "Dummy" else grids[name]
        candidates = []
        for params in ParameterGrid(grid):
            inner_scores = []
            for inner_fold, (inner_train, inner_valid) in enumerate(inner_cv.split(X_train, y_train), 1):
                preprocessor, encoded_train, encoded_valid, mdi_record = prepare_split(X_train.iloc[inner_train], y_train[inner_train], X_train.iloc[inner_valid])
                model = make_estimator(name, params, y_fit=y_train[inner_train])
                if name == "OPF":
                    model.fit(np.asarray(encoded_train), np.asarray(y_train[inner_train], dtype=np.int32))
                    prediction = model.predict(np.asarray(encoded_valid))
                else:
                    model.fit(encoded_train, y_train[inner_train])
                    prediction = model.predict(encoded_valid)
                inner_scores.append(f1_score(y_train[inner_valid], prediction, labels=[0, 1], average="macro", zero_division=0))
                phase1_search.append({"scenario": "baseline", "model": name, "outer_fold": outer_fold, "inner_fold": inner_fold, "params": dict(params), "params_applied": model.get_params(deep=True) if hasattr(model, "get_params") else dict(params), "f1_macro": inner_scores[-1]})
            candidates.append((float(np.mean(inner_scores)), dict(params)))
        best_f1, best_params = max(candidates, key=lambda item: item[0])
        preprocessor, encoded_train, encoded_test, mdi_record = prepare_split(X_train, y_train, X_test)
        model = make_estimator(name, best_params, y_fit=y_train)
        if name == "OPF":
            model.fit(np.asarray(encoded_train), np.asarray(y_train, dtype=np.int32))
            prediction, score0, score1, score_source = predict_with_score(model, np.asarray(encoded_test))
        else:
            model.fit(encoded_train, y_train)
            prediction, score0, score1, score_source = predict_with_score(model, encoded_test)
        metrics = binary_metrics(y_test, prediction, score0, score1)
        metrics.update({"scenario": "baseline", "model": name, "outer_fold": outer_fold, "params": best_params, "inner_f1_macro": best_f1})
        phase1_rows.append(metrics)
        phase1_oof[name][test_index] = prediction
        if score1 is not None:
            phase1_scores[name][test_index] = score1
        print("Fase 1", name, metrics)


Fase 1 KNN {'accuracy': 0.75, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8059701492537313), 'recall_0': np.float64(0.0), 'recall_1': np.float64(0.9152542372881356), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.8571428571428571), 'f1_macro': 0.42857142857142855, 'f1_weighted': 0.7023809523809523, 'precision_macro': 0.40298507462686567, 'recall_macro': 0.4576271186440678, 'balanced_accuracy': 0.4576271186440678, 'mcc': -0.12823107147006824, 'specificity': np.float64(0.0), 'sensitivity': np.float64(0.9152542372881356), 'TN': 0, 'FP': 13, 'FN': 5, 'TP': 54, 'score_source': 'continuous', 'auroc': 0.4902216427640157, 'ap_0': 0.17858044615260238, 'ap_1': 0.8226043170507016, 'scenario': 'baseline', 'model': 'KNN', 'outer_fold': 1, 'params': {'n_neighbors': 5, 'weights': 'distance'}, 'inner_f1_macro': 0.48583631986178144}
Fase 1 DT {'accuracy': 0.6527777777777778, 'precision_0': np.float64(0.16666666666666666), 'precision_1': np.float64(0.8148148148148148), 'recall_0': np.floa

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 1 MLP {'accuracy': 0.8194444444444444, 'precision_0': np.float64(0.5), 'precision_1': np.float64(0.8285714285714286), 'recall_0': np.float64(0.07692307692307693), 'recall_1': np.float64(0.9830508474576272), 'f1_0': np.float64(0.13333333333333333), 'f1_1': np.float64(0.8992248062015504), 'f1_macro': 0.5162790697674419, 'f1_weighted': 0.7609388458225668, 'precision_macro': 0.6642857142857144, 'recall_macro': 0.529986962190352, 'balanced_accuracy': 0.529986962190352, 'mcc': 0.14037705656838215, 'specificity': np.float64(0.07692307692307693), 'sensitivity': np.float64(0.9830508474576272), 'TN': 1, 'FP': 12, 'FN': 1, 'TP': 58, 'score_source': 'continuous', 'auroc': 0.634941329856584, 'ap_0': 0.29282719695105053, 'ap_1': 0.9090614668759465, 'scenario': 'baseline', 'model': 'MLP', 'outer_fold': 1, 'params': {'alpha': 0.01, 'hidden_layer_sizes': (32,), 'max_iter': 500}, 'inner_f1_macro': 0.4795592417052627}
Fase 1 LR {'accuracy': 0.8194444444444444, 'precision_0': np.float64(0.0), 'precis

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/ma

Fase 1 NC {'accuracy': 0.5555555555555556, 'precision_0': np.float64(0.21212121212121213), 'precision_1': np.float64(0.8461538461538461), 'recall_0': np.float64(0.5384615384615384), 'recall_1': np.float64(0.559322033898305), 'f1_0': np.float64(0.30434782608695654), 'f1_1': np.float64(0.673469387755102), 'f1_macro': 0.4889086069210293, 'f1_weighted': 0.6068224391205758, 'precision_macro': 0.5291375291375291, 'recall_macro': 0.5488917861799217, 'balanced_accuracy': 0.5488917861799217, 'mcc': 0.07548737230565958, 'specificity': np.float64(0.5384615384615384), 'sensitivity': np.float64(0.559322033898305), 'TN': 7, 'FP': 6, 'FN': 26, 'TP': 33, 'score_source': 'continuous', 'auroc': 0.6179921773142113, 'ap_0': 0.28290442114371184, 'ap_1': 0.8925504134673372, 'scenario': 'baseline', 'model': 'NC', 'outer_fold': 1, 'params': {'metric': 'euclidean', 'shrink_threshold': None}, 'inner_f1_macro': 0.45892838431981176}
2026-09-13 19:40:20,326 - opfython.models.supervised — INFO — Overriding class: O

...

Fase 1 MLP {'accuracy': 0.8028169014084507, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8142857142857143), 'recall_0': np.float64(0.0), 'recall_1': np.float64(0.9827586206896551), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.890625), 'f1_macro': 0.4453125, 'f1_weighted': 0.7275528169014085, 'precision_macro': 0.40714285714285714, 'recall_macro': 0.49137931034482757, 'balanced_accuracy': 0.49137931034482757, 'mcc': -0.056585956237831254, 'specificity': np.float64(0.0), 'sensitivity': np.float64(0.9827586206896551), 'TN': 0, 'FP': 13, 'FN': 1, 'TP': 57, 'score_source': 'continuous', 'auroc': 0.4933687002652519, 'ap_0': 0.19287913555492714, 'ap_1': 0.8195926605595842, 'scenario': 'baseline', 'model': 'MLP', 'outer_fold': 10, 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32,), 'max_iter': 500}, 'inner_f1_macro': 0.4663610010436865}
Fase 1 LR {'accuracy': 0.8169014084507042, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8169014084507042), 'recall_0': np.f

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/ma

Fase 1 NC {'accuracy': 0.4788732394366197, 'precision_0': np.float64(0.18421052631578946), 'precision_1': np.float64(0.8181818181818182), 'recall_0': np.float64(0.5384615384615384), 'recall_1': np.float64(0.46551724137931033), 'f1_0': np.float64(0.27450980392156865), 'f1_1': np.float64(0.5934065934065934), 'f1_macro': 0.43395819866408103, 'f1_weighted': 0.5350170404022931, 'precision_macro': 0.5011961722488039, 'recall_macro': 0.5019893899204244, 'balanced_accuracy': 0.5019893899204244, 'mcc': 0.0030852247988512774, 'specificity': np.float64(0.5384615384615384), 'sensitivity': np.float64(0.46551724137931033), 'TN': 7, 'FP': 6, 'FN': 31, 'TP': 27, 'score_source': 'unavailable', 'auroc': nan, 'ap_0': nan, 'ap_1': nan, 'scenario': 'baseline', 'model': 'NC', 'outer_fold': 10, 'params': {'metric': 'manhattan', 'shrink_threshold': 0.1}, 'inner_f1_macro': 0.4574688950181591}
2026-09-13 19:46:30,668 - opfython.models.supervised — INFO — Overriding class: OPF -> SupervisedOPF.
2026-09-13 19:46:

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neighbors/_nearest_centroid.py:241: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(
/home/pk/Studing/projects/ma

2026-09-13 19:46:30,978 - opfython.models.supervised — INFO — Classifier has been fitted.
2026-09-13 19:46:30,979 - opfython.models.supervised — INFO — Training time: 0.30768752098083496 seconds.
2026-09-13 19:46:30,979 - opfython.models.supervised — INFO — Predicting data ...
2026-09-13 19:46:31,064 - opfython.models.supervised — INFO — Data has been predicted.
2026-09-13 19:46:31,066 - opfython.models.supervised — INFO — Prediction time: 0.08572912216186523 seconds.
2026-09-13 19:46:31,076 - opfython.models.supervised — INFO — Overriding class: OPF -> SupervisedOPF.
2026-09-13 19:46:31,077 - opfython.core.opf — INFO — Creating class: OPF.
2026-09-13 19:46:31,077 - opfython.core.opf — DEBUG — Distance: euclidean | Pre-computed distance: False.
2026-09-13 19:46:31,078 - opfython.core.opf — INFO — Class created.
2026-09-13 19:46:31,078 - opfython.models.supervised — INFO — Class overrided.
2026-09-13 19:46:31,079 - opfython.models.supervised — INFO — Fitting classifier ...
2026-09-13 19

# Fase 2 — MDI 80%
MDI é ajustado somente no treino de cada split, após transformação de todas as features.


In [95]:
phase2_rows = []
phase2_oof = {name: np.full(len(y_array), -1, dtype=int) for name in roster}
phase2_scores = {name: np.full(len(y_array), np.nan) for name in roster}
phase2_mdi = []
for outer_fold, (train_index, test_index) in enumerate(outer_cv.split(X, y_array), 1):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y_array[train_index]
    y_test = y_array[test_index]
    for name in roster:
        candidates = []
        for params in ParameterGrid(grids[name]):
            inner_scores = []
            for inner_fold, (inner_train, inner_valid) in enumerate(inner_cv.split(X_train, y_train), 1):
                preprocessor, encoded_train, encoded_valid, mdi_record = prepare_split(X_train.iloc[inner_train], y_train[inner_train], X_train.iloc[inner_valid], use_mdi=True)
                model = make_estimator(name, params, y_fit=y_train[inner_train])
                if name == "OPF":
                    model.fit(np.asarray(encoded_train), np.asarray(y_train[inner_train], dtype=np.int32))
                    prediction = model.predict(np.asarray(encoded_valid))
                else:
                    model.fit(encoded_train, y_train[inner_train])
                    prediction = model.predict(encoded_valid)
                inner_scores.append(f1_score(y_train[inner_valid], prediction, labels=[0, 1], average="macro", zero_division=0))
                phase2_mdi.append({"scenario": "mdi80", "model": name, "outer_fold": outer_fold, "inner_fold": inner_fold, **mdi_record})
            candidates.append((float(np.mean(inner_scores)), dict(params)))
        best_f1, best_params = max(candidates, key=lambda item: item[0])
        preprocessor, encoded_train, encoded_test, mdi_record = prepare_split(X_train, y_train, X_test, use_mdi=True)
        model = make_estimator(name, best_params, y_fit=y_train)
        if name == "OPF":
            model.fit(np.asarray(encoded_train), np.asarray(y_train, dtype=np.int32))
            prediction, score0, score1, _ = predict_with_score(model, np.asarray(encoded_test))
        else:
            model.fit(encoded_train, y_train)
            prediction, score0, score1, _ = predict_with_score(model, encoded_test)
        metrics = binary_metrics(y_test, prediction, score0, score1)
        metrics.update({"scenario": "mdi80", "model": name, "outer_fold": outer_fold, "params": best_params, "inner_f1_macro": best_f1})
        phase2_rows.append(metrics)
        phase2_oof[name][test_index] = prediction
        if score1 is not None:
            phase2_scores[name][test_index] = score1
        phase2_mdi.append({"scenario": "mdi80", "model": name, "outer_fold": outer_fold, "inner_fold": None, **mdi_record})
        print("Fase 2", name, metrics)


Fase 2 KNN {'accuracy': 0.7916666666666666, 'precision_0': np.float64(0.3333333333333333), 'precision_1': np.float64(0.8333333333333334), 'recall_0': np.float64(0.15384615384615385), 'recall_1': np.float64(0.9322033898305084), 'f1_0': np.float64(0.21052631578947367), 'f1_1': np.float64(0.88), 'f1_macro': 0.5452631578947369, 'f1_weighted': 0.7591228070175439, 'precision_macro': 0.5833333333333334, 'recall_macro': 0.5430247718383312, 'balanced_accuracy': 0.5430247718383312, 'mcc': 0.11975638025916219, 'specificity': np.float64(0.15384615384615385), 'sensitivity': np.float64(0.9322033898305084), 'TN': 2, 'FP': 11, 'FN': 4, 'TP': 55, 'score_source': 'continuous', 'auroc': 0.4250325945241199, 'ap_0': 0.22562017927871586, 'ap_1': 0.7813306434746965, 'scenario': 'mdi80', 'model': 'KNN', 'outer_fold': 1, 'params': {'n_neighbors': 5, 'weights': 'distance'}, 'inner_f1_macro': 0.5167853027815077}
Fase 2 DT {'accuracy': 0.7777777777777778, 'precision_0': np.float64(0.2857142857142857), 'precision_

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 2 MLP {'accuracy': 0.8194444444444444, 'precision_0': np.float64(0.5), 'precision_1': np.float64(0.8285714285714286), 'recall_0': np.float64(0.07692307692307693), 'recall_1': np.float64(0.9830508474576272), 'f1_0': np.float64(0.13333333333333333), 'f1_1': np.float64(0.8992248062015504), 'f1_macro': 0.5162790697674419, 'f1_weighted': 0.7609388458225668, 'precision_macro': 0.6642857142857144, 'recall_macro': 0.529986962190352, 'balanced_accuracy': 0.529986962190352, 'mcc': 0.14037705656838215, 'specificity': np.float64(0.07692307692307693), 'sensitivity': np.float64(0.9830508474576272), 'TN': 1, 'FP': 12, 'FN': 1, 'TP': 58, 'score_source': 'continuous', 'auroc': 0.6544980443285529, 'ap_0': 0.3302065766644683, 'ap_1': 0.898460737014892, 'scenario': 'mdi80', 'model': 'MLP', 'outer_fold': 1, 'params': {'alpha': 0.01, 'hidden_layer_sizes': (32, 16), 'max_iter': 500}, 'inner_f1_macro': 0.4844097387384384}
Fase 2 LR {'accuracy': 0.8194444444444444, 'precision_0': np.float64(0.0), 'precisi

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 2 MLP {'accuracy': 0.7638888888888888, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8088235294117647), 'recall_0': np.float64(0.0), 'recall_1': np.float64(0.9322033898305084), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.8661417322834646), 'f1_macro': 0.4330708661417323, 'f1_weighted': 0.7097550306211723, 'precision_macro': 0.40441176470588236, 'recall_macro': 0.4661016949152542, 'balanced_accuracy': 0.4661016949152542, 'mcc': -0.1138469000458504, 'specificity': np.float64(0.0), 'sensitivity': np.float64(0.9322033898305084), 'TN': 0, 'FP': 13, 'FN': 4, 'TP': 55, 'score_source': 'continuous', 'auroc': 0.4374185136897001, 'ap_0': 0.1688367347974419, 'ap_1': 0.8055773166709896, 'scenario': 'mdi80', 'model': 'MLP', 'outer_fold': 2, 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32, 16), 'max_iter': 500}, 'inner_f1_macro': 0.49111862629865144}
Fase 2 LR {'accuracy': 0.8194444444444444, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8194444444444444), 'r

...

Fase 2 MLP {'accuracy': 0.8309859154929577, 'precision_0': np.float64(1.0), 'precision_1': np.float64(0.8285714285714286), 'recall_0': np.float64(0.07692307692307693), 'recall_1': np.float64(1.0), 'f1_0': np.float64(0.14285714285714285), 'f1_1': np.float64(0.90625), 'f1_macro': 0.5245535714285714, 'f1_weighted': 0.7664738430583501, 'precision_macro': 0.9142857142857144, 'recall_macro': 0.5384615384615384, 'balanced_accuracy': 0.5384615384615384, 'mcc': 0.25246042013801634, 'specificity': np.float64(0.07692307692307693), 'sensitivity': np.float64(1.0), 'TN': 1, 'FP': 12, 'FN': 0, 'TP': 58, 'score_source': 'continuous', 'auroc': 0.6677718832891246, 'ap_0': 0.3411262225057754, 'ap_1': 0.9029259935920334, 'scenario': 'mdi80', 'model': 'MLP', 'outer_fold': 8, 'params': {'alpha': 0.01, 'hidden_layer_sizes': (32, 16), 'max_iter': 500}, 'inner_f1_macro': 0.47060380367188304}
Fase 2 LR {'accuracy': 0.8169014084507042, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8169014084507042)

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 2 MLP {'accuracy': 0.7323943661971831, 'precision_0': np.float64(0.2), 'precision_1': np.float64(0.819672131147541), 'recall_0': np.float64(0.15384615384615385), 'recall_1': np.float64(0.8620689655172413), 'f1_0': np.float64(0.17391304347826086), 'f1_1': np.float64(0.8403361344537815), 'f1_macro': 0.5071245889660212, 'f1_weighted': 0.7183150051202354, 'precision_macro': 0.5098360655737705, 'recall_macro': 0.5079575596816976, 'balanced_accuracy': 0.5079575596816976, 'mcc': 0.01769418874505073, 'specificity': np.float64(0.15384615384615385), 'sensitivity': np.float64(0.8620689655172413), 'TN': 2, 'FP': 11, 'FN': 8, 'TP': 50, 'score_source': 'continuous', 'auroc': 0.4190981432360743, 'ap_0': 0.1745239338365939, 'ap_1': 0.7769877468139312, 'scenario': 'mdi80', 'model': 'MLP', 'outer_fold': 9, 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32, 16), 'max_iter': 500}, 'inner_f1_macro': 0.4867797031671316}
Fase 2 LR {'accuracy': 0.8169014084507042, 'precision_0': np.float64(0.0), 'pre

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 2 MLP {'accuracy': 0.8309859154929577, 'precision_0': np.float64(1.0), 'precision_1': np.float64(0.8285714285714286), 'recall_0': np.float64(0.07692307692307693), 'recall_1': np.float64(1.0), 'f1_0': np.float64(0.14285714285714285), 'f1_1': np.float64(0.90625), 'f1_macro': 0.5245535714285714, 'f1_weighted': 0.7664738430583501, 'precision_macro': 0.9142857142857144, 'recall_macro': 0.5384615384615384, 'balanced_accuracy': 0.5384615384615384, 'mcc': 0.25246042013801634, 'specificity': np.float64(0.07692307692307693), 'sensitivity': np.float64(1.0), 'TN': 1, 'FP': 12, 'FN': 0, 'TP': 58, 'score_source': 'continuous', 'auroc': 0.5716180371352785, 'ap_0': 0.28306314801908994, 'ap_1': 0.87787612756409, 'scenario': 'mdi80', 'model': 'MLP', 'outer_fold': 10, 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32,), 'max_iter': 500}, 'inner_f1_macro': 0.47010796420262296}
Fase 2 LR {'accuracy': 0.8169014084507042, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8169014084507042),

# Fase 3 — Custo sensível
Pesos calculados exclusivamente com labels do treino corrente. KNN, NB, NC e OPF não são aplicáveis.


In [96]:
cost_roster = ["DT", "RF", "SVM", "LR", "XGB"]
phase3_rows = []
phase3_oof = {name: np.full(len(y_array), -1, dtype=int) for name in cost_roster}
phase3_scores = {name: np.full(len(y_array), np.nan) for name in cost_roster}
phase3_weights = []
phase3_non_applicable = {name: "não aplicável" for name in ["KNN", "NB", "NC", "OPF"]}
for name, reason in phase3_non_applicable.items():
    print("Fase 3", name, reason)
for outer_fold, (train_index, test_index) in enumerate(outer_cv.split(X, y_array), 1):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y_array[train_index]
    y_test = y_array[test_index]
    outer_weights = prevalence_weights(y_train)
    phase3_weights.append({"outer_fold": outer_fold, "weights": outer_weights, "y_fit_indices": train_index.tolist()})
    for name in cost_roster:
        grid = grids[name]
        candidates = []
        for params in ParameterGrid(grid):
            inner_scores = []
            for inner_fold, (inner_train, inner_valid) in enumerate(inner_cv.split(X_train, y_train), 1):
                y_fit = y_train[inner_train]
                preprocessor, encoded_train, encoded_valid, _ = prepare_split(X_train.iloc[inner_train], y_fit, X_train.iloc[inner_valid])
                model = make_estimator(name, params, scenario="cost_sensitive", y_fit=y_fit)
                model.fit(encoded_train, y_fit)
                prediction = model.predict(encoded_valid)
                inner_scores.append(f1_score(y_train[inner_valid], prediction, labels=[0, 1], average="macro", zero_division=0))
            candidates.append((float(np.mean(inner_scores)), dict(params)))
        best_f1, best_params = max(candidates, key=lambda item: item[0])
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        model = make_estimator(name, best_params, scenario="cost_sensitive", y_fit=y_train)
        model.fit(encoded_train, y_train)
        prediction, score0, score1, _ = predict_with_score(model, encoded_test)
        metrics = binary_metrics(y_test, prediction, score0, score1)
        metrics.update({"scenario": "cost_sensitive", "model": name, "outer_fold": outer_fold, "params": best_params, "inner_f1_macro": best_f1})
        phase3_rows.append(metrics)
        phase3_oof[name][test_index] = prediction
        if score1 is not None:
            phase3_scores[name][test_index] = score1
        print("Fase 3", name, metrics)


Fase 3 KNN não aplicável
Fase 3 NB não aplicável
Fase 3 NC não aplicável
Fase 3 OPF não aplicável
Fase 3 DT {'accuracy': 0.6944444444444444, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.7936507936507936), 'recall_0': np.float64(0.0), 'recall_1': np.float64(0.847457627118644), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.819672131147541), 'f1_macro': 0.4098360655737705, 'f1_weighted': 0.6716757741347905, 'precision_macro': 0.3968253968253968, 'recall_macro': 0.423728813559322, 'balanced_accuracy': 0.423728813559322, 'mcc': -0.1774175796776985, 'specificity': np.float64(0.0), 'sensitivity': np.float64(0.847457627118644), 'TN': 0, 'FP': 13, 'FN': 9, 'TP': 50, 'score_source': 'continuous', 'auroc': 0.423728813559322, 'ap_0': 0.18055555555555555, 'ap_1': 0.7965177517376796, 'scenario': 'cost_sensitive', 'model': 'DT', 'outer_fold': 1, 'params': {'max_depth': None, 'min_samples_leaf': 1}, 'inner_f1_macro': 0.5137522833997994}
Fase 3 RF {'accuracy': 0.7083333333333334, 'precis

# Fase 4 — Ensemble
Membros são selecionados por perfis OOF internos, qualidade e diversidade real de erros.


In [98]:
ensemble_candidates = ["KNN", "DT", "RF", "SVM", "MLP", "LR", "XGB", "NB", "AdaBoost"]
ensemble_excluded = {"NC": "sem predict_proba/decision_function", "OPF": "sem predict_proba/decision_function"}
phase4_manifest = {"phase": "ensemble", "excluded_candidates": ensemble_excluded, "members_by_fold": []}
phase4_rows = []
phase4_oof_classes = pd.DataFrame({"y_true": y_array})
phase4_oof_scores = pd.DataFrame({"y_true": y_array})
phase4_inner_oof = []
phase4_profiles = []
phase4_members = []

def ensemble_model(name, params):
    if name not in ensemble_candidates:
        raise ValueError(f"Modelo não calibrável para ensemble probabilístico: {name}")
    base = make_estimator(name, params)
    if name == "SVM":
        return base
    return CalibratedClassifierCV(estimator=base, cv=5, method="sigmoid")


def candidate_inner_profile(X_fit, y_fit, name, params):
    truth = []
    predictions = []
    scores0 = []
    scores1 = []
    indices = []
    for inner_fold, (inner_train, inner_valid) in enumerate(inner_cv.split(X_fit, y_fit), 1):
        preprocessor, encoded_train, encoded_valid, _ = prepare_split(X_fit.iloc[inner_train], y_fit[inner_train], X_fit.iloc[inner_valid])
        model = ensemble_model(name, params)
        model.fit(encoded_train, y_fit[inner_train])
        prediction, score0, score1, score_source = predict_with_score(model, encoded_valid)
        y_outer_train = y_fit
        truth.extend(y_outer_train[inner_valid])
        predictions.extend(prediction)
        indices.extend(inner_valid.tolist())
        if score1 is not None:
            scores0.extend(score0.tolist())
            scores1.extend(score1.tolist())
    truth = np.asarray(truth)
    predictions = np.asarray(predictions)
    score0 = np.asarray(scores0) if scores0 else None
    score1 = np.asarray(scores1) if scores1 else None
    metrics = binary_metrics(truth, predictions, score0, score1)
    metrics.update({
        "model": name,
        "params": dict(params),
        "indices": indices,
        "y_true": truth.tolist(),
        "y_pred": predictions.tolist(),
        "score": score1.tolist() if score1 is not None else None,
        "predictions": predictions.tolist(),
        "errors": (predictions != truth).astype(int).tolist(),
        "score0": score0.tolist() if score0 is not None else None,
        "score1": score1.tolist() if score1 is not None else None,
        "score_source": "predict_proba" if score1 is not None else "unavailable",
    })
    return metrics


for outer_fold, (train_index, test_index) in enumerate(outer_cv.split(X, y_array), 1):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y_array[train_index]
    y_test = y_array[test_index]
    profiles = []

    params_KNN = []
    f1_val_KNN = []
    acc_val_KNN = []
    for params in ParameterGrid(param_grid_KNN):
        profile = candidate_inner_profile(X_train, y_train, "KNN", params)
        params_KNN.append(dict(params))
        f1_val_KNN.append(profile["f1_macro"])
        acc_val_KNN.append(profile["accuracy"])
    best_params_KNN = params_KNN[int(np.argmax(f1_val_KNN))]
    profile_KNN = candidate_inner_profile(X_train, y_train, "KNN", best_params_KNN)
    profiles.append(profile_KNN)
    print("Fase 4 KNN perfil interno", profile_KNN)

    params_DT = []
    f1_val_DT = []
    acc_val_DT = []
    for params in ParameterGrid(param_grid_DT):
        profile = candidate_inner_profile(X_train, y_train, "DT", params)
        params_DT.append(dict(params))
        f1_val_DT.append(profile["f1_macro"])
        acc_val_DT.append(profile["accuracy"])
    best_params_DT = params_DT[int(np.argmax(f1_val_DT))]
    profile_DT = candidate_inner_profile(X_train, y_train, "DT", best_params_DT)
    profiles.append(profile_DT)
    print("Fase 4 DT perfil interno", profile_DT)

    params_RF = []
    f1_val_RF = []
    acc_val_RF = []
    for params in ParameterGrid(param_grid_RF):
        profile = candidate_inner_profile(X_train, y_train, "RF", params)
        params_RF.append(dict(params))
        f1_val_RF.append(profile["f1_macro"])
        acc_val_RF.append(profile["accuracy"])
    best_params_RF = params_RF[int(np.argmax(f1_val_RF))]
    profile_RF = candidate_inner_profile(X_train, y_train, "RF", best_params_RF)
    profiles.append(profile_RF)
    print("Fase 4 RF perfil interno", profile_RF)

    params_SVM = []
    f1_val_SVM = []
    acc_val_SVM = []
    for params in ParameterGrid(param_grid_SVM):
        profile = candidate_inner_profile(X_train, y_train, "SVM", params)
        params_SVM.append(dict(params))
        f1_val_SVM.append(profile["f1_macro"])
        acc_val_SVM.append(profile["accuracy"])
    best_params_SVM = params_SVM[int(np.argmax(f1_val_SVM))]
    profile_SVM = candidate_inner_profile(X_train, y_train, "SVM", best_params_SVM)
    profiles.append(profile_SVM)
    print("Fase 4 SVM perfil interno", profile_SVM)

    params_MLP = []
    f1_val_MLP = []
    acc_val_MLP = []
    for params in ParameterGrid(param_grid_MLP):
        profile = candidate_inner_profile(X_train, y_train, "MLP", params)
        params_MLP.append(dict(params))
        f1_val_MLP.append(profile["f1_macro"])
        acc_val_MLP.append(profile["accuracy"])
    best_params_MLP = params_MLP[int(np.argmax(f1_val_MLP))]
    profile_MLP = candidate_inner_profile(X_train, y_train, "MLP", best_params_MLP)
    profiles.append(profile_MLP)
    print("Fase 4 MLP perfil interno", profile_MLP)

    params_LR = []
    f1_val_LR = []
    acc_val_LR = []
    for params in ParameterGrid(param_grid_LR):
        profile = candidate_inner_profile(X_train, y_train, "LR", params)
        params_LR.append(dict(params))
        f1_val_LR.append(profile["f1_macro"])
        acc_val_LR.append(profile["accuracy"])
    best_params_LR = params_LR[int(np.argmax(f1_val_LR))]
    profile_LR = candidate_inner_profile(X_train, y_train, "LR", best_params_LR)
    profiles.append(profile_LR)
    print("Fase 4 LR perfil interno", profile_LR)

    params_XGB = []
    f1_val_XGB = []
    acc_val_XGB = []
    for params in ParameterGrid(param_grid_XGB):
        profile = candidate_inner_profile(X_train, y_train, "XGB", params)
        params_XGB.append(dict(params))
        f1_val_XGB.append(profile["f1_macro"])
        acc_val_XGB.append(profile["accuracy"])
    best_params_XGB = params_XGB[int(np.argmax(f1_val_XGB))]
    profile_XGB = candidate_inner_profile(X_train, y_train, "XGB", best_params_XGB)
    profiles.append(profile_XGB)
    print("Fase 4 XGB perfil interno", profile_XGB)

    params_NB = []
    f1_val_NB = []
    acc_val_NB = []
    for params in ParameterGrid(param_grid_NB):
        profile = candidate_inner_profile(X_train, y_train, "NB", params)
        params_NB.append(dict(params))
        f1_val_NB.append(profile["f1_macro"])
        acc_val_NB.append(profile["accuracy"])
    best_params_NB = params_NB[int(np.argmax(f1_val_NB))]
    profile_NB = candidate_inner_profile(X_train, y_train, "NB", best_params_NB)
    profiles.append(profile_NB)
    print("Fase 4 NB perfil interno", profile_NB)

    params_AdaBoost = []
    f1_val_AdaBoost = []
    acc_val_AdaBoost = []
    for params in ParameterGrid(param_grid_AdaBoost):
        profile = candidate_inner_profile(X_train, y_train, "AdaBoost", params)
        params_AdaBoost.append(dict(params))
        f1_val_AdaBoost.append(profile["f1_macro"])
        acc_val_AdaBoost.append(profile["accuracy"])
    best_params_AdaBoost = params_AdaBoost[int(np.argmax(f1_val_AdaBoost))]
    profile_AdaBoost = candidate_inner_profile(X_train, y_train, "AdaBoost", best_params_AdaBoost)
    profiles.append(profile_AdaBoost)
    print("Fase 4 AdaBoost perfil interno", profile_AdaBoost)

    ranked = sorted(profiles, key=lambda item: (item["f1_macro"], item["f1_0"], item["recall_0"], item["balanced_accuracy"], item["mcc"], item["ap_1"]), reverse=True)
    selected = [ranked[0]]
    diversity_matrix = {}
    while len(selected) < 3:
        remaining = [item for item in ranked if item not in selected]
        for candidate in remaining:
            diversity_matrix[candidate["model"]] = [
                float(np.mean(np.asarray(candidate["errors"]) != np.asarray(member["errors"])))
                for member in selected
            ]
        selected.append(max(remaining, key=lambda item: (0.8 * item["f1_macro"] + 0.2 * np.mean(diversity_matrix[item["model"]]), item["f1_0"], item["recall_0"], item["balanced_accuracy"], item["mcc"], item["ap_1"])))
    selected_names = [item["model"] for item in selected]
    assert len(selected_names) == 3
    assert len(set(selected_names)) == 3
    weights = np.ones(3, dtype=float) / 3
    member_models = {}
    member_predictions = {}
    member_score0 = {}
    member_score1 = {}
    selected_params = {item["model"]: item["params"] for item in selected}
    if "KNN" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["KNN"] = ensemble_model("KNN", selected_params["KNN"])
        member_models["KNN"].fit(encoded_train, y_train)
        member_predictions["KNN"], member_score0["KNN"], member_score1["KNN"], _ = predict_with_score(member_models["KNN"], encoded_test)
        print("Fase 4 membro KNN avaliado")
    if "DT" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["DT"] = ensemble_model("DT", selected_params["DT"])
        member_models["DT"].fit(encoded_train, y_train)
        member_predictions["DT"], member_score0["DT"], member_score1["DT"], _ = predict_with_score(member_models["DT"], encoded_test)
        print("Fase 4 membro DT avaliado")
    if "RF" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["RF"] = ensemble_model("RF", selected_params["RF"])
        member_models["RF"].fit(encoded_train, y_train)
        member_predictions["RF"], member_score0["RF"], member_score1["RF"], _ = predict_with_score(member_models["RF"], encoded_test)
        print("Fase 4 membro RF avaliado")
    if "SVM" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["SVM"] = ensemble_model("SVM", selected_params["SVM"])
        member_models["SVM"].fit(encoded_train, y_train)
        member_predictions["SVM"], member_score0["SVM"], member_score1["SVM"], _ = predict_with_score(member_models["SVM"], encoded_test)
        print("Fase 4 membro SVM avaliado")
    if "MLP" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["MLP"] = ensemble_model("MLP", selected_params["MLP"])
        member_models["MLP"].fit(encoded_train, y_train)
        member_predictions["MLP"], member_score0["MLP"], member_score1["MLP"], _ = predict_with_score(member_models["MLP"], encoded_test)
        print("Fase 4 membro MLP avaliado")
    if "LR" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["LR"] = ensemble_model("LR", selected_params["LR"])
        member_models["LR"].fit(encoded_train, y_train)
        member_predictions["LR"], member_score0["LR"], member_score1["LR"], _ = predict_with_score(member_models["LR"], encoded_test)
        print("Fase 4 membro LR avaliado")
    if "XGB" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["XGB"] = ensemble_model("XGB", selected_params["XGB"])
        member_models["XGB"].fit(encoded_train, y_train)
        member_predictions["XGB"], member_score0["XGB"], member_score1["XGB"], _ = predict_with_score(member_models["XGB"], encoded_test)
        print("Fase 4 membro XGB avaliado")
    if "NB" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["NB"] = ensemble_model("NB", selected_params["NB"])
        member_models["NB"].fit(encoded_train, y_train)
        member_predictions["NB"], member_score0["NB"], member_score1["NB"], _ = predict_with_score(member_models["NB"], encoded_test)
        print("Fase 4 membro NB avaliado")
    if "AdaBoost" in selected_names:
        preprocessor, encoded_train, encoded_test, _ = prepare_split(X_train, y_train, X_test)
        member_models["AdaBoost"] = ensemble_model("AdaBoost", selected_params["AdaBoost"])
        member_models["AdaBoost"].fit(encoded_train, y_train)
        member_predictions["AdaBoost"], member_score0["AdaBoost"], member_score1["AdaBoost"], _ = predict_with_score(member_models["AdaBoost"], encoded_test)
        print("Fase 4 membro AdaBoost avaliado")
    predictions = [member_predictions[name] for name in selected_names]
    scores0 = [member_score0[name] for name in selected_names]
    scores1 = [member_score1[name] for name in selected_names]
    majority = (np.sum(predictions, axis=0) >= 2).astype(int)
    probability_score0 = np.average(np.vstack(scores0), axis=0, weights=weights)
    probability_score1 = np.average(np.vstack(scores1), axis=0, weights=weights)
    probability = (probability_score1 >= 0.5).astype(int)
    majority_metrics = binary_metrics(y_test, majority, probability_score0, probability_score1)
    probability_metrics = binary_metrics(y_test, probability, probability_score0, probability_score1)
    phase4_oof_classes.loc[test_index, "majority_class"] = majority
    phase4_oof_classes.loc[test_index, "probability_class"] = probability
    phase4_oof_scores.loc[test_index, "majority_score0"] = probability_score0
    phase4_oof_scores.loc[test_index, "majority_score1"] = probability_score1
    phase4_oof_scores.loc[test_index, "probability_score0"] = probability_score0
    phase4_oof_scores.loc[test_index, "probability_score1"] = probability_score1
    inner_probability = np.mean(np.vstack([np.asarray(item["score1"]) for item in selected]), axis=0)
    phase4_inner_oof.append({
        "outer_fold": outer_fold,
        "indices": selected[0]["indices"],
        "y_true": selected[0]["y_true"],
        "y_pred": (inner_probability >= 0.5).astype(int).tolist(),
        "score": inner_probability.tolist(),
        "members": selected_names,
    })
    member_record = {"outer_fold": outer_fold, "members": selected_names, "weights": weights.tolist(), "profiles": profiles, "diversity_matrix": diversity_matrix, "majority_metrics": majority_metrics, "probability_metrics": probability_metrics, "excluded_candidates": ensemble_excluded}
    phase4_members.append(member_record)
    phase4_manifest["members_by_fold"].append(member_record)
    phase4_profiles.extend([{**profile, "outer_fold": outer_fold} for profile in profiles])
    phase4_rows.extend([{**majority_metrics, "scenario": "ensemble_majority", "outer_fold": outer_fold}, {**probability_metrics, "scenario": "ensemble_probability", "outer_fold": outer_fold}])


Fase 4 KNN perfil interno {'accuracy': 0.8161993769470405, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8161993769470405), 'recall_0': np.float64(0.0), 'recall_1': np.float64(1.0), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.8987993138936535), 'f1_macro': 0.44939965694682676, 'f1_weighted': 0.7335994400004274, 'precision_macro': 0.40809968847352024, 'recall_macro': 0.5, 'balanced_accuracy': 0.5, 'mcc': 0.0, 'specificity': np.float64(0.0), 'sensitivity': np.float64(1.0), 'TN': 0, 'FP': 118, 'FN': 0, 'TP': 524, 'score_source': 'predict_proba', 'auroc': 0.466514102729978, 'ap_0': 0.16847767862280497, 'ap_1': 0.7950644810812072, 'model': 'KNN', 'params': {'n_neighbors': 5, 'weights': 'uniform'}, 'indices': [2, 5, 13, 15, 19, 38, 42, 43, 45, 46, 47, 48, 49, 54, 58, 61, 68, 77, 79, 82, 83, 86, 89, 91, 92, 94, 95, 96, 98, 106, 107, 108, 110, 114, 117, 119, 120, 133, 155, 157, 161, 163, 171, 174, 182, 188, 191, 192, 194, 197, 200, 204, 206, 213, 217, 220, 224, 227, 228, 231, 2

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 4 MLP perfil interno {'accuracy': 0.8161993769470405, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8161993769470405), 'recall_0': np.float64(0.0), 'recall_1': np.float64(1.0), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.8987993138936535), 'f1_macro': 0.44939965694682676, 'f1_weighted': 0.7335994400004274, 'precision_macro': 0.40809968847352024, 'recall_macro': 0.5, 'balanced_accuracy': 0.5, 'mcc': 0.0, 'specificity': np.float64(0.0), 'sensitivity': np.float64(1.0), 'TN': 0, 'FP': 118, 'FN': 0, 'TP': 524, 'score_source': 'predict_proba', 'auroc': 0.4611932332772674, 'ap_0': 0.16835518593426488, 'ap_1': 0.7951824211032998, 'model': 'MLP', 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32,), 'max_iter': 500}, 'indices': [2, 5, 13, 15, 19, 38, 42, 43, 45, 46, 47, 48, 49, 54, 58, 61, 68, 77, 79, 82, 83, 86, 89, 91, 92, 94, 95, 96, 98, 106, 107, 108, 110, 114, 117, 119, 120, 133, 155, 157, 161, 163, 171, 174, 182, 188, 191, 192, 194, 197, 200, 204, 206, 213, 217, 22

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 4 MLP perfil interno {'accuracy': 0.8161993769470405, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8161993769470405), 'recall_0': np.float64(0.0), 'recall_1': np.float64(1.0), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.8987993138936535), 'f1_macro': 0.44939965694682676, 'f1_weighted': 0.7335994400004274, 'precision_macro': 0.40809968847352024, 'recall_macro': 0.5, 'balanced_accuracy': 0.5, 'mcc': 0.0, 'specificity': np.float64(0.0), 'sensitivity': np.float64(1.0), 'TN': 0, 'FP': 118, 'FN': 0, 'TP': 524, 'score_source': 'predict_proba', 'auroc': 0.48162763617544313, 'ap_0': 0.18370325218314082, 'ap_1': 0.808831537396015, 'model': 'MLP', 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32,), 'max_iter': 500}, 'indices': [3, 5, 13, 15, 20, 40, 43, 44, 46, 47, 48, 49, 50, 57, 59, 61, 63, 75, 81, 82, 85, 86, 88, 91, 92, 94, 95, 98, 99, 101, 102, 106, 110, 114, 116, 118, 119, 132, 143, 159, 164, 166, 171, 173, 182, 187, 189, 190, 195, 200, 203, 207, 209, 212, 217, 22

...

Fase 4 MLP perfil interno {'accuracy': 0.8164852255054432, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8164852255054432), 'recall_0': np.float64(0.0), 'recall_1': np.float64(1.0), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.898972602739726), 'f1_macro': 0.449486301369863, 'f1_weighted': 0.7339978482711604, 'precision_macro': 0.4082426127527216, 'recall_macro': 0.5, 'balanced_accuracy': 0.5, 'mcc': 0.0, 'specificity': np.float64(0.0), 'sensitivity': np.float64(1.0), 'TN': 0, 'FP': 118, 'FN': 0, 'TP': 525, 'score_source': 'predict_proba', 'auroc': 0.3907506053268765, 'ap_0': 0.14563347314906785, 'ap_1': 0.7661733587561409, 'model': 'MLP', 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32,), 'max_iter': 500}, 'indices': [2, 4, 12, 14, 18, 38, 42, 43, 45, 46, 47, 48, 49, 54, 57, 60, 67, 76, 79, 81, 82, 84, 88, 89, 90, 92, 93, 95, 102, 105, 109, 112, 114, 115, 116, 128, 133, 141, 151, 157, 159, 165, 168, 169, 177, 182, 185, 187, 189, 192, 195, 202, 205, 208, 211, 214, 

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 4 MLP perfil interno {'accuracy': 0.8164852255054432, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8164852255054432), 'recall_0': np.float64(0.0), 'recall_1': np.float64(1.0), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.898972602739726), 'f1_macro': 0.449486301369863, 'f1_weighted': 0.7339978482711604, 'precision_macro': 0.4082426127527216, 'recall_macro': 0.5, 'balanced_accuracy': 0.5, 'mcc': 0.0, 'specificity': np.float64(0.0), 'sensitivity': np.float64(1.0), 'TN': 0, 'FP': 118, 'FN': 0, 'TP': 525, 'score_source': 'predict_proba', 'auroc': 0.5044632768361582, 'ap_0': 0.1887731752564532, 'ap_1': 0.8260539516085412, 'model': 'MLP', 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32,), 'max_iter': 500}, 'indices': [2, 5, 13, 15, 19, 39, 42, 44, 46, 47, 48, 49, 50, 55, 59, 62, 64, 71, 78, 82, 83, 85, 90, 91, 92, 93, 94, 95, 97, 106, 107, 109, 111, 113, 116, 118, 120, 132, 146, 158, 162, 164, 172, 174, 183, 188, 191, 192, 194, 197, 200, 204, 206, 212, 216, 219, 22

/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pk/Studing/projects/machine_learning/Artigo_veis/analyze-medical-services/.venv/li

Fase 4 MLP perfil interno {'accuracy': 0.8164852255054432, 'precision_0': np.float64(0.0), 'precision_1': np.float64(0.8164852255054432), 'recall_0': np.float64(0.0), 'recall_1': np.float64(1.0), 'f1_0': np.float64(0.0), 'f1_1': np.float64(0.898972602739726), 'f1_macro': 0.449486301369863, 'f1_weighted': 0.7339978482711604, 'precision_macro': 0.4082426127527216, 'recall_macro': 0.5, 'balanced_accuracy': 0.5, 'mcc': 0.0, 'specificity': np.float64(0.0), 'sensitivity': np.float64(1.0), 'TN': 0, 'FP': 118, 'FN': 0, 'TP': 525, 'score_source': 'predict_proba', 'auroc': 0.4432687651331719, 'ap_0': 0.16198845590501748, 'ap_1': 0.7937950720535627, 'model': 'MLP', 'params': {'alpha': 0.0001, 'hidden_layer_sizes': (32,), 'max_iter': 500}, 'indices': [2, 5, 13, 15, 19, 38, 42, 43, 45, 46, 47, 48, 49, 54, 58, 62, 67, 77, 79, 82, 83, 86, 89, 91, 92, 93, 95, 96, 98, 106, 107, 108, 110, 114, 117, 119, 121, 133, 143, 159, 163, 165, 174, 177, 184, 189, 192, 193, 195, 198, 201, 208, 211, 214, 217, 220, 2

# Fase 5 — Limiar e consolidação
O limiar é escolhido somente em OOF interno do ensemble calibrado e aplicado ao mesmo score externo da Fase 4.


In [99]:
phase5_rows = []
phase5_thresholds = []
phase5_oof_classes = pd.DataFrame({"y_true": y_array})
phase5_oof_scores = pd.DataFrame({"y_true": y_array})
threshold_grid = np.arange(0.10, 0.91, 0.05)
for outer_fold, (train_index, test_index) in enumerate(outer_cv.split(X, y_array), 1):
    internal = phase4_inner_oof[outer_fold - 1]
    inner_truth = np.asarray(internal["y_true"])
    inner_indices = np.asarray(internal["indices"])
    inner_score1 = np.asarray(internal["score"], dtype=float)
    inner_prediction = np.asarray(internal["y_pred"])
    assert len(inner_indices) == len(inner_truth) == len(inner_prediction) == len(inner_score1)
    assert not pd.isna(inner_indices).any()
    assert len(np.unique(inner_indices)) == len(inner_indices)
    assert np.isfinite(inner_score1).all()
    threshold_records = []
    for threshold in threshold_grid:
        inner_prediction = (inner_score1 >= threshold).astype(int)
        threshold_records.append({
            "threshold": float(threshold),
            "f1_macro": f1_score(inner_truth, inner_prediction, labels=[0, 1], average="macro", zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(inner_truth, inner_prediction),
            "mcc": matthews_corrcoef(inner_truth, inner_prediction),
        })
    chosen = max(threshold_records, key=lambda row: (row["f1_macro"], row["balanced_accuracy"], row["mcc"]))
    threshold = chosen["threshold"]
    outer_score1 = phase4_oof_scores.loc[test_index, "probability_score1"].to_numpy(dtype=float)
    outer_score0 = phase4_oof_scores.loc[test_index, "probability_score0"].to_numpy(dtype=float)
    assert np.isfinite(outer_score1).all()
    y_test = y_array[test_index]
    prediction_05 = (outer_score1 >= 0.5).astype(int)
    prediction_threshold = (outer_score1 >= threshold).astype(int)
    metrics_05 = binary_metrics(y_test, prediction_05, outer_score0, outer_score1)
    metrics_threshold = binary_metrics(y_test, prediction_threshold, outer_score0, outer_score1)
    phase5_oof_classes.loc[test_index, "threshold_05_class"] = prediction_05
    phase5_oof_classes.loc[test_index, "threshold_selected_class"] = prediction_threshold
    phase5_oof_scores.loc[test_index, "score0"] = outer_score0
    phase5_oof_scores.loc[test_index, "score1"] = outer_score1
    phase5_thresholds.append({
        "outer_fold": outer_fold,
        "threshold": threshold,
        "inner_oof_size": len(inner_truth),
        "members": internal["members"],
        "selection": threshold_records,
    })
    phase5_rows.extend([
        {**metrics_05, "scenario": "threshold_0.5", "outer_fold": outer_fold, "threshold": 0.5},
        {**metrics_threshold, "scenario": "threshold_selected", "outer_fold": outer_fold, "threshold": threshold},
    ])
    print("Fase 5 fold", outer_fold, "threshold", threshold)


Fase 5 fold 1 threshold 0.8000000000000002
Fase 5 fold 2 threshold 0.8000000000000002
Fase 5 fold 3 threshold 0.8000000000000002
Fase 5 fold 4 threshold 0.8000000000000002
Fase 5 fold 5 threshold 0.8000000000000002
Fase 5 fold 6 threshold 0.8000000000000002
Fase 5 fold 7 threshold 0.8000000000000002
Fase 5 fold 8 threshold 0.8000000000000002
Fase 5 fold 9 threshold 0.8000000000000002
Fase 5 fold 10 threshold 0.8000000000000002


# Consolidação e exportação
Exportação executada somente quando este notebook for executado. O diretório é separado dos resultados legados.


In [104]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
artifact_root = Path("src/artifacts/revisao_experimental") / run_id
fold_metrics_df = pd.DataFrame(phase1_rows + phase2_rows + phase3_rows + phase4_rows + phase5_rows)
search_df = pd.DataFrame(phase1_search)
folds_df = pd.DataFrame(fold_indices)
mdi_df = pd.DataFrame(phase2_mdi)
weights_df = pd.DataFrame(phase3_weights)
ensemble_profiles_df = pd.DataFrame(phase4_profiles)
ensemble_members_df = pd.DataFrame(phase4_members)
thresholds_df = pd.DataFrame(phase5_thresholds)
assert len(folds_df) == 10
assert all(set(row["train_index"]).isdisjoint(row["test_index"]) for row in fold_indices)
assert all(len(values) == 714 and np.all(np.asarray(values) != -1) for values in phase1_oof.values())
assert all(len(values) == 714 and np.all(np.asarray(values) != -1) for values in phase2_oof.values())
assert all(len(values) == 714 and np.all(np.asarray(values) != -1) for values in phase3_oof.values())
assert phase4_oof_classes["majority_class"].notna().sum() == 714
assert phase4_oof_classes["probability_class"].notna().sum() == 714
assert phase5_oof_classes["threshold_05_class"].notna().sum() == 714
assert phase5_oof_classes["threshold_selected_class"].notna().sum() == 714
assert phase5_oof_scores["score1"].notna().sum() == 714
assert phase5_oof_scores["score0"].notna().sum() == 714
assert all(len(internal["indices"]) == len(set(internal["indices"])) for internal in phase4_inner_oof)
assert all(len(internal["indices"]) == len(internal["y_true"]) == len(internal["score"]) for internal in phase4_inner_oof)
assert all(np.isfinite(np.asarray(internal["score"], dtype=float)).all() for internal in phase4_inner_oof)
score_availability = []
for scenario, score_map in [("baseline", phase1_scores), ("mdi80", phase2_scores), ("cost_sensitive", phase3_scores)]:
    for model_name, score_values in score_map.items():
        finite = np.isfinite(score_values)
        missing = np.isnan(score_values)
        if model_name == "OPF":
            assert missing.all()
            status, reason = "all_nan", "OPF sem score contínuo"
        elif finite.all():
            status, reason = "all_finite", None
        elif missing.all():
            status, reason = "all_nan", "helper registrou score indisponível"
        else:
            score_map[model_name] = np.full_like(score_values, np.nan, dtype=float)
            status, reason = "partial_score_suppressed", "mistura de scores finitos e NaN"
        score_availability.append({"scenario": scenario, "model": model_name, "status": status, "reason": reason})

assert all(len(record["members"]) == 3 and len(set(record["members"])) == 3 for record in phase4_members)
assert all(abs(sum(record["weights"]) - 1.0) < 1e-12 for record in phase4_members)
assert all(0.10 <= record["threshold"] <= 0.90 for record in phase5_thresholds)
mdi_frequency_df = pd.DataFrame(columns=["feature_origin", "selection_frequency", "total_folds"])
if not mdi_df.empty:
    selected_rows = []
    for record in phase2_mdi:
        mdi_records = record.get("mdi_records") or []
        assert all({"feature_transformed", "feature_origin", "importance", "cumulative", "selected"}.issubset(item) for item in mdi_records)
        selected_rows.extend(
            {**item, "outer_fold": record["outer_fold"]}
            for item in mdi_records
        )
    selection_table = pd.DataFrame(selected_rows)
    if not selection_table.empty:
        selected_only = selection_table[selection_table["selected"]]
        mdi_frequency_df = selected_only.groupby("feature_origin").agg(selection_frequency=("selected", "sum"), total_folds=("outer_fold", "nunique")).reset_index()

wilcoxon_rows = []
from scipy.stats import wilcoxon
for model in roster:
    corrected = fold_metrics_df[(fold_metrics_df["scenario"] == "mdi80") & (fold_metrics_df["model"] == model)].drop_duplicates("outer_fold").set_index("outer_fold")["f1_macro"]
    dummy = fold_metrics_df[(fold_metrics_df["scenario"] == "baseline") & (fold_metrics_df["model"] == "Dummy")].drop_duplicates("outer_fold").set_index("outer_fold")["f1_macro"]
    pairs = pd.concat([corrected, dummy], axis=1, join="inner").dropna()
    if len(pairs) == 0 or np.allclose(pairs.iloc[:, 0], pairs.iloc[:, 1]):
        statistic = 0.0
        p_value = 1.0
    else:
        statistic, p_value = wilcoxon(pairs.iloc[:, 0], pairs.iloc[:, 1])
    difference = pairs.iloc[:, 0] - pairs.iloc[:, 1] if len(pairs) else pd.Series(dtype=float)
    wilcoxon_rows.append({"contrast": model + " vs Dummy", "W": float(statistic), "p": float(p_value), "mean_difference": float(difference.mean()) if len(difference) else np.nan, "n_folds": len(pairs), "limitation": "10 folds dependentes, exploratório"})
order = np.argsort([row["p"] for row in wilcoxon_rows])
running = 0.0
adjusted = np.zeros(len(wilcoxon_rows), dtype=float)
for rank, position in enumerate(order):
    running = max(running, min(1.0, (len(order) - rank) * wilcoxon_rows[position]["p"]))
    adjusted[position] = min(1.0, running)
for position, row in enumerate(wilcoxon_rows):
    row["p_holm"] = min(1.0, float(adjusted[position]))
assert all(0 <= row["p_holm"] <= 1 for row in wilcoxon_rows)

manifest = {"run_id": run_id, "seed": SEED, "n": len(y_array), "target_mapping": TARGET_MAP, "class_counts": class_counts.to_dict(), "class_percentages": class_percentages.to_dict(), "data_path": str(DATA_PATH), "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(), "nested_cv": {"outer": 10, "inner": 5, "shuffle": True}, "fold_indices": fold_indices, "roster": roster + ["AdaBoost", "Dummy"], "versions": {"python": sys.version, "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__}, "limitations": ["n=714", "desenho transversal", "base única", "sem validação externa", "generalização limitada"], "opf_score_note": "OPF sem score contínuo mantém AUROC/AP como NaN"}
manifest["phase4_ensemble"] = phase4_manifest
manifest["score_availability"] = score_availability
artifact_root.mkdir(parents=True, exist_ok=False)
folds_df.to_csv(artifact_root / "folds.csv", index=False)
fold_metrics_df.to_csv(artifact_root / "metrics_per_fold.csv", index=False)
search_df.to_csv(artifact_root / "search.csv", index=False)
mdi_df.to_json(artifact_root / "mdi_detailed.json", orient="records", indent=2)
mdi_frequency_df.to_csv(artifact_root / "mdi_frequency.csv", index=False)
weights_df.to_json(artifact_root / "weights.json", orient="records", indent=2)
ensemble_profiles_df.to_json(artifact_root / "ensemble_profiles.json", orient="records", indent=2)
ensemble_members_df.to_json(artifact_root / "ensemble_members.json", orient="records", indent=2)
thresholds_df.to_json(artifact_root / "thresholds.json", orient="records", indent=2)
pd.DataFrame(wilcoxon_rows).to_json(artifact_root / "wilcoxon_holm.json", orient="records", indent=2)
phase4_oof_classes.to_csv(artifact_root / "oof_ensemble_classes.csv", index=False)
phase4_oof_scores.to_csv(artifact_root / "oof_ensemble_scores.csv", index=False)
phase5_oof_classes.to_csv(artifact_root / "oof_threshold_classes.csv", index=False)
phase5_oof_scores.to_csv(artifact_root / "oof_threshold_scores.csv", index=False)
for name, values in phase1_oof.items():
    if np.isfinite(phase1_scores[name]).all():
        pd.DataFrame({"y_true": y_array, "prediction": values, "score1": phase1_scores[name]}).to_csv(artifact_root / f"oof_baseline_{name}.csv", index=False)
for name, values in phase2_oof.items():
    if np.isfinite(phase2_scores[name]).all():
        pd.DataFrame({"y_true": y_array, "prediction": values, "score1": phase2_scores[name]}).to_csv(artifact_root / f"oof_mdi80_{name}.csv", index=False)
for name, values in phase3_oof.items():
    if np.isfinite(phase3_scores[name]).all():
        pd.DataFrame({"y_true": y_array, "prediction": values, "score1": phase3_scores[name]}).to_csv(artifact_root / f"oof_cost_{name}.csv", index=False)
for scenario, scores in [("baseline", phase1_scores), ("mdi80", phase2_scores), ("cost_sensitive", phase3_scores)]:
    for name, values in scores.items():
        valid = np.isfinite(values)
        if valid.all():
            fpr, tpr, _ = roc_curve(y_array, values)
            pd.DataFrame({"fpr": fpr, "tpr": tpr}).to_csv(artifact_root / f"roc_{scenario}_{name}.csv", index=False)
            precision, recall, _ = precision_recall_curve(y_array, values)
            pd.DataFrame({"precision": precision, "recall": recall}).to_csv(artifact_root / f"pr_{scenario}_{name}.csv", index=False)
plot_sources = [("baseline", phase1_scores), ("mdi80", phase2_scores), ("cost_sensitive", phase3_scores)]
for scenario, score_map in plot_sources:
    for model_name, score_values in score_map.items():
        valid = np.isfinite(score_values)
        if valid.all():
            false_positive_rate, true_positive_rate, _ = roc_curve(y_array, score_values)
            figure, axis = plt.subplots()
            axis.plot(false_positive_rate, true_positive_rate)
            axis.set_title(f"ROC — {scenario} — {model_name} — seed {SEED}")
            axis.set_xlabel("False positive rate")
            axis.set_ylabel("True positive rate")
            figure.savefig(artifact_root / f"roc_{scenario}_{model_name}.png", bbox_inches="tight")
            plt.close(figure)
            precision, recall, _ = precision_recall_curve(y_array, score_values)
            figure, axis = plt.subplots()
            axis.plot(recall, precision)
            axis.set_title(f"PR — {scenario} — {model_name} — seed {SEED}")
            axis.set_xlabel("Recall")
            axis.set_ylabel("Precision")
            figure.savefig(artifact_root / f"pr_{scenario}_{model_name}.png", bbox_inches="tight")
            plt.close(figure)
manifest["artifacts"] = sorted(str(path.relative_to(artifact_root)) for path in artifact_root.iterdir())
(artifact_root / "manifest.json").write_text(json.dumps({**manifest, "artifact_root": str(artifact_root)}, ensure_ascii=False, indent=2))
print("Artefatos exportados em", artifact_root)


Artefatos exportados em src/artifacts/revisao_experimental/20260914T000127Z
